# M6.4 Two-Dimensional FDTD: Point Sources, Energy Flow and Diffraction

Book pointer: chapter 17, section 17.2 (Maxwell's equations). The 2D Yee scheme itself is taught in primer 00d of this module.

Notebook 03 sent pulses along a line. Here the same leapfrog runs on a plane, which brings in what one dimension cannot show: waves that spread out, energy that flows in a direction, and waves that bend around obstacles. Every result is checked against a number you can work out by hand.

## 1. Recall

Answer on paper before running any cell.

1. Write the three equations of the 2D TM system ($E_z$, $H_x$, $H_y$) from Faraday's law and the Ampere-Maxwell law. Which spatial derivative drives which component?
2. What is the Courant limit for a square 2D Yee grid, and why is it smaller than in 1D?
3. Why does the amplitude of a cylindrical wave fall as $1/\sqrt{r}$ far from a line source? Use energy flowing through circles of growing radius.
4. For $\mathbf{E} = E_z\hat{\mathbf{z}}$ and $\mathbf{H}$ in the $xy$ plane, write both components of $\mathbf{S} = \mathbf{E}\times\mathbf{H}$.
5. A layer with electric conductivity $\sigma$ and magnetic conductivity $\sigma_m = \sigma\mu_0/\epsilon_0$ has the impedance of vacuum. Why does a normally incident wave enter it without reflecting, and why does it then die out?

## 2. Principle to algorithm

When nothing depends on $z$, Maxwell's curl equations split into two independent sets. The TM set keeps $E_z$, $H_x$ and $H_y$:

$$\mu_0\frac{\partial H_x}{\partial t} = -\frac{\partial E_z}{\partial y},\qquad \mu_0\frac{\partial H_y}{\partial t} = \frac{\partial E_z}{\partial x},\qquad \epsilon\frac{\partial E_z}{\partial t} = \frac{\partial H_y}{\partial x} - \frac{\partial H_x}{\partial y} - \sigma E_z.$$

The Yee grid puts each field where its derivatives are centred: $E_z$ on the nodes $(i, j)$, $H_x$ on $(i, j+\tfrac12)$, $H_y$ on $(i+\tfrac12, j)$, with $H$ half a time step out of step with $E$. Every derivative is then one difference across one cell. `emlib.fdtd.fdtd_2d_tm` advances $H$ from the differences of $E_z$, then $E_z$ from the circulation of $H$ around each node. The step obeys $S = c\,\Delta t/\Delta x \le 1/\sqrt2$: in 2D a disturbance reaches a node from two directions at once, so the step must be shorter than in 1D.

Three more pieces finish the method.

- **The frame.** The grid ends in a PEC wall. In front of it, `n_layer` cells carry a conductivity that rises as the cube of depth, with a magnetic loss matched to it, $\sigma_m = \sigma\mu_0/\epsilon_0$. A wave entering head-on sees no change of impedance and decays as it goes in and out again. At oblique incidence and on a grid some reflection remains, which section 4 measures.
- **The source.** A soft source adds a value to $E_z$ at one node each step. That is a line current along $z$, so it radiates a cylindrical wave. A column of such nodes is a current sheet and radiates a plane wave.
- **Measurement.** Probes record $E_z$ at chosen nodes every step, and snapshots keep whole maps. `emlib.fdtd.poynting_2d_tm` averages $H$ onto the nodes and returns $S_x = -E_zH_y$ and $S_y = E_zH_x$. The half-step offset between $E$ and $H$ is ignored, which costs little for anything integrated over a pulse.

Two experiments need what `fdtd_2d_tm` does not expose: the fields at every step, and a metal obstacle inside the grid. For those the notebook writes out the same update with two hooks added, and checks that it reproduces the library to the last bit.

## 3. Build

### 3.1 A point source

A Gaussian pulse injected at the centre of a 301 by 301 grid with $\Delta x = 1$ mm. The four snapshots show a ring expanding at $c$. Behind the front the field does not return to zero at once: in two dimensions a pulse leaves a slowly decaying tail, because the line source's contributions from different heights along $z$ keep arriving. That tail is why waves in 2D behave differently from 1D and 3D, and it matters for timing below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from scipy.signal import find_peaks
from emlib import fdtd as fd
from emlib import fields
from emlib.fields import EPS0
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 5)

n, dx, S = 301, 1e-3, 0.5
dt = S * dx / fd.C0
w = 20 * dt
t0 = 4 * w
pulse = lambda t: fd.gaussian(t, t0, w)
c0 = (150, 150)

radii = np.array([20, 30, 40, 60, 80, 100])
axis_probes = [(c0[0] + int(r), c0[1]) for r in radii]
diag_probes = [(c0[0] + round(r / np.sqrt(2)), c0[1] + round(r / np.sqrt(2))) for r in radii]
run = fd.fdtd_2d_tm(n, n, dx, 500, courant=S, source=pulse, src_index=c0,
                    probes=axis_probes + diag_probes, snapshot_every=100)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, k in zip(axes, range(4)):
    snap = run["snapshots"][k]
    lim = np.abs(snap).max()
    ax.imshow(snap.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim, extent=[0, 300, 0, 300])
    ax.add_patch(plt.Rectangle((20, 20), 260, 260, fill=False, ls=":", color="k"))
    ax.set_title(f"step {100 * (k + 1)}  (t = {100 * (k + 1) * dt * 1e12:.0f} ps)")
    ax.set_xlabel("x (cells)")
axes[0].set_ylabel("y (cells)")
fig.suptitle("Ez of a point source; dotted square: inner edge of the absorbing frame")
plt.tight_layout(); plt.show()

In [ ]:
t, pe = run["t"], run["probe_E"]
k_ax = len(radii)
t_axis = np.array([t[np.argmax(pe[:, k])] for k in range(k_ax)])
t_diag = np.array([t[np.argmax(pe[:, k_ax + k])] for k in range(k_ax)])
r_diag = np.array([np.hypot(i - c0[0], j - c0[1]) for i, j in diag_probes])
amp_axis = pe[:, :k_ax].max(axis=0)
amp_diag = pe[:, k_ax:].max(axis=0)

print(" r (cells)  peak step  r/c in steps  early by (cells)")
for r, ta in zip(radii, t_axis):
    print(f"{r:8d}   {ta / dt:8.0f}   {(t0 + r * dx / fd.C0) / dt:10.0f}   {(t0 + r * dx / fd.C0 - ta) * fd.C0 / dx:12.1f}")
v = (80 - 40) * dx / (t_axis[4] - t_axis[2])
print(f"\nspeed from the delay between r = 40 and r = 80 cells: {v / fd.C0:.4f} c")
print("diagonal probes: radius", r_diag.round(2), "\n  peak step minus axis peak step:", ((t_diag - t_axis) / dt).astype(int),
      "\n  expected from the radius difference:", ((r_diag - radii) * dx / fd.C0 / dt).round(1))
slope = np.polyfit(np.log(radii), np.log(amp_axis), 1)[0]
print(f"\npeak amplitude ratio r = 40 to r = 80: {amp_axis[2] / amp_axis[4]:.4f}   sqrt(2) = {np.sqrt(2):.4f}")
print(f"log-log slope of peak amplitude against r: {slope:.3f}")
print("diagonal / axis peak amplitude:", (amp_diag / amp_axis).round(3))

plt.loglog(radii, amp_axis, "o", label="axis probes")
plt.loglog(r_diag, amp_diag, "s", mfc="none", label="diagonal probes")
plt.loglog(radii, amp_axis[2] * np.sqrt(40 / radii), "k--", label=r"$\propto 1/\sqrt{r}$")
plt.xlabel("r (cells)"); plt.ylabel("peak Ez (V/m)"); plt.legend(); plt.title("cylindrical spreading")
plt.show()

Read the numbers.

- **Timing.** The peak reaches each probe four to five cells earlier than $t_0 + r/c$, by the same amount at every radius. The waveform of a 2D pulse is not the source pulse shifted: the tail drags its shape, and the peak sits ahead of where a delayed copy would put it. A constant offset cancels in a difference, so the speed comes from the delay between two radii, and it is $c$ to four digits.
- **Isotropy.** The diagonal probes sit at slightly different radii because the grid has no node at exactly $r/\sqrt2$ along both axes. Allowing for that, they arrive within one step of the axis probes. Their amplitude is about one percent higher: the Yee grid carries diagonal waves slightly better than axial ones at this resolution.
- **Spreading.** Energy flows out through circles of circumference $2\pi r$, so the intensity falls as $1/r$ and the amplitude as $1/\sqrt r$. The measured slope, $-0.48$, approaches $-1/2$ from above because the $1/\sqrt r$ law is the far-field limit and the nearest probes are only a few pulse widths from the source.

### 3.2 Energy flow

The Poynting vector says where electromagnetic energy goes. For the TM fields it lies in the plane, and for an outgoing wave it points away from the source. Two checks of energy conservation are possible on the grid.

1. Once the source has switched off and before the pulse reaches the frame, the total field energy per unit length along $z$, $U = \sum \tfrac12(\epsilon E_z^2 + \mu_0 H_x^2 + \mu_0 H_y^2)\,\Delta x^2$, must stay constant.
2. All of that energy eventually crosses any closed curve around the source, so the time integral of the outward flux of $\mathbf{S}$ through a ring must equal $U$.

The second check needs $H$ at every step. `fdtd_2d_tm` takes a `callback(k, Ez, Hx, Hy)` that it calls after every step, with $E_z$ at time $k\Delta t$ and $H_x, H_y$ half a step earlier, and an optional `pec` mask of perfectly conducting nodes, which section 3.3 uses for the slit walls.

### Exercise: the flux through a ring

Replace the body of `ring_flux` with your own version. It takes the two components of a vector field sampled at $(i\Delta x, j\Delta x)$ and returns the outward flux through a circle, $\oint \mathbf{S}\cdot\hat{\mathbf{n}}\,dl$, by sampling the field at points around the circle with linear interpolation. The check uses a field whose flux is known exactly, $\mathbf{S} = (\mathbf{r} - \mathbf{r}_c)/|\mathbf{r} - \mathbf{r}_c|^2$, which carries $2\pi$ through every closed curve around $\mathbf{r}_c$, and compares the ring with the square box of `emlib.fields.box_flux`.

In [ ]:
def ring_flux(Sx, Sy, centre, radius, dx):
    '''Outward flux of (Sx, Sy), sampled at (i dx, j dx), through a circle of the given centre and radius.'''
    m = int(np.ceil(8 * np.pi * radius / dx))                      # replace from here with your own version
    phi = (np.arange(m) + 0.5) * 2 * np.pi / m
    px = (centre[0] + radius * np.cos(phi)) / dx
    py = (centre[1] + radius * np.sin(phi)) / dx
    sx = map_coordinates(Sx, [px, py], order=1)
    sy = map_coordinates(Sy, [px, py], order=1)
    return float(np.sum(sx * np.cos(phi) + sy * np.sin(phi)) * radius * 2 * np.pi / m)


xs = np.arange(201) * dx
Xs, Ys = np.meshgrid(xs, xs, indexing="ij")
rc = (100 * dx, 100 * dx)
r2 = (Xs - rc[0]) ** 2 + (Ys - rc[1]) ** 2
r2[100, 100] = 1.0
Sx_test, Sy_test = (Xs - rc[0]) / r2, (Ys - rc[1]) / r2
f_ring = ring_flux(Sx_test, Sy_test, rc, 0.05, dx)
f_box = fields.box_flux(np.stack([Sx_test, Sy_test], axis=-1), (dx, dx), (50, 50), (150, 150))
print(f"ring: {f_ring:.6f}   box: {f_box:.6f}   2 pi = {2 * np.pi:.6f}")
assert abs(f_ring - 2 * np.pi) < 1e-3 and abs(f_ring - f_box) < 1e-3

In [ ]:
# The point source again, now with the energy and the flux recorded every step.
# poynting_2d_tm returns arrays on interior nodes, so its index i sits at Ez index i + 1.
ring_r = 50
flux_ring, flux_box, U = [], [], {}
snapshot = {}

def record(k, Ez, Hx, Hy):
    Sx, Sy = fd.poynting_2d_tm(Ez, Hx, Hy)
    centre = ((c0[0] - 1) * dx, (c0[1] - 1) * dx)
    flux_ring.append(ring_flux(Sx, Sy, centre, ring_r * dx, dx))
    lo, hi = (c0[0] - 1 - ring_r, c0[1] - 1 - ring_r), (c0[0] - 1 + ring_r, c0[1] - 1 + ring_r)
    flux_box.append(fields.box_flux(np.stack([Sx, Sy], axis=-1), (dx, dx), lo, hi))
    if k in (170, 180, 200, 220):
        U[k] = 0.5 * dx**2 * (EPS0 * np.sum(Ez**2) + MU0 * (np.sum(Hx**2) + np.sum(Hy**2)))
    if k == 200:
        snapshot.update(Ez=Ez.copy(), Sx=Sx.copy(), Sy=Sy.copy())

fd.fdtd_2d_tm(n, n, dx, 1200, source=pulse, src_index=c0, callback=record)
flux_ring, flux_box = np.array(flux_ring), np.array(flux_box)
steps = np.arange(1, 1201)

print("field energy per unit length (J/m) after the source has stopped:")
for k, u in U.items():
    print(f"  step {k}: {u:.5e}")
E_ring, E_box = flux_ring.sum() * dt, flux_box.sum() * dt
print(f"time-integrated flux through the ring: {E_ring:.5e} J/m  ({E_ring / U[200]:.4f} of the field energy)")
print(f"time-integrated flux through the box:  {E_box:.5e} J/m  ({E_box / U[200]:.4f})")
print(f"most negative instantaneous ring flux, relative to the peak: {flux_ring.min() / flux_ring.max():.1e}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
Ez200 = snapshot["Ez"][1:-1, 1:-1]
axes[0].imshow(np.abs(Ez200).T, origin="lower", cmap="magma", extent=[1, 300, 1, 300])
st = 10
I, J = np.meshgrid(np.arange(1, 300), np.arange(1, 300), indexing="ij")
Smag = np.hypot(snapshot["Sx"], snapshot["Sy"])
keep = Smag[::st, ::st] > 0.05 * Smag.max()
axes[0].quiver(I[::st, ::st][keep], J[::st, ::st][keep], snapshot["Sx"][::st, ::st][keep], snapshot["Sy"][::st, ::st][keep], color="c")
axes[0].add_patch(plt.Circle(c0, ring_r, fill=False, color="w", ls="--"))
axes[0].set_xlim(60, 240); axes[0].set_ylim(60, 240)
axes[0].set_title("|Ez| at step 200 and the Poynting vector (arrows)")
axes[1].plot(steps, flux_ring / flux_ring.max(), label="flux through the ring (normalised)")
axes[1].plot(steps, np.cumsum(flux_ring) * dt / U[200], label="time integral / field energy")
axes[1].axhline(1, color="k", lw=0.5)
axes[1].set_xlabel("step"); axes[1].legend(); axes[1].set_title(f"energy leaving a ring of radius {ring_r} cells")
plt.tight_layout(); plt.show()

The field energy is constant to four digits between steps 170 and 220, after the source has stopped and before the pulse reaches the frame. The time integral of the flux through the ring recovers it to within half a percent, and the square box agrees with the ring. The small shortfall is the price of the staggered time levels: $E$ and $H$ are half a step apart, so both the energy sum and the instantaneous Poynting vector are defined only to second order in $\Delta t$. The flux through the ring never turns negative beyond a few parts in a thousand, so no energy comes back from the frame.

### 3.3 Diffraction by one slit and by two

A plane wave falls on a metal wall with openings and the intensity is recorded on a screen behind it. The wall is a block of PEC nodes, where $E_z$ is held at zero. The source is a column of soft sources carrying a sine at $\lambda = 10$ mm, switched on smoothly so that the start does not ring. Once the pattern has settled, the intensity is the time average of $E_z^2$ over the last ten periods.

The grid uses $\Delta x = 0.5$ mm, twenty cells per wavelength; the reason for that choice appears at the end of this section. The screen sits $D = 0.298$ m behind the wall. The absorbing frame is 80 cells, four wavelengths, thick. With a frame half as thick, weak reflections put a ripple on the screen that splits the single slit's first zero into two minima a few millimetres apart: a zero is where any small stray wave shows most.

Two textbook formulas describe the patterns, both derived for the far field:

- **one slit of width $a$:** intensity zeros where $a\sin\theta = m\lambda$;
- **two narrow slits a distance $d$ apart:** zeros where $d\sin\theta = (m + \tfrac12)\lambda$. At small angles the fringes repeat every $\lambda D/d$ on the screen.

Three approximations separate those formulas from this simulation. The geometry is two-dimensional, so the slits are infinitely long slots and each one radiates a cylindrical wave. The screen is at a finite distance: $D$ is about three times $a^2/\lambda$ for the single slit and about $1.2\,d^2/\lambda$ for the double slit, so the far-field condition holds only roughly for the second. And the angles are not small, so $\lambda D/d$ is the small-angle spacing, while the exact position of each zero follows from the path difference between the two slits to the screen point. The table below reports all three numbers.

In [ ]:
def slit_experiment(slits, dx, lam, D_cells, wall_cells, n_layer, avg_periods=10, ny=None):
    '''Plane wave on a PEC wall with slits [(centre offset in cells, width in cells), ...].
    Returns the screen coordinate y (m), the mean of Ez^2 on the screen, and the final Ez map.'''
    dt = 0.5 * dx / fd.C0
    f = fd.C0 / lam
    T = 1 / f
    i_src, i_wall = n_layer + 20, n_layer + 60
    i_exit = i_wall + wall_cells - 1
    i_scr = i_exit + D_cells
    nx = i_scr + n_layer + 40
    ny = ny or nx
    jc = ny // 2
    pec = np.zeros((nx, ny), dtype=bool)
    pec[i_wall:i_wall + wall_cells, :] = True
    for cj, wc in slits:
        j0 = jc + cj - wc // 2
        pec[i_wall:i_wall + wall_cells, j0:j0 + wc] = False
    line = (np.full(ny - 2, i_src), np.arange(1, ny - 1))
    sine = lambda t: 0.01 * np.sin(2 * np.pi * f * t) * (1 - np.exp(-(t / (3 * T)) ** 2))
    n_steps = int(round((i_scr - i_src + 0.6 * ny) / 0.5)) + int(round(avg_periods * T / dt))
    start = n_steps - int(round(avg_periods * T / dt))
    acc = np.zeros(ny)
    count = [0]
    def collect(k, Ez, Hx, Hy):
        if k > start:
            acc[:] += Ez[i_scr, :] ** 2
            count[0] += 1
    Ez = fd.fdtd_2d_tm(nx, ny, dx, n_steps, source=sine, src_index=line, n_layer=n_layer, pec=pec, callback=collect)["Ez"]
    y = (np.arange(ny) - jc) * dx
    return y, acc / count[0], Ez, dict(D=D_cells * dx, i_wall=i_wall, i_scr=i_scr, n_steps=n_steps)


def zeros_of(I, y):
    '''Intensity minima, refined by a parabola through each minimum and its neighbours.
    Shallow ripples (small reflections from the frame) are skipped by requiring real depth.
    Returns (negative side, innermost first), (positive side, innermost first).'''
    out = []
    for k in find_peaks(-I, prominence=0.02 * I.max())[0]:
        a, b, c = I[k - 1], I[k], I[k + 1]
        out.append(y[k] + 0.5 * (a - c) / (a - 2 * b + c) * (y[1] - y[0]))
    out = np.array(out)
    return np.sort(out[out < 0])[::-1], np.sort(out[out > 0])


dxs, lam = 0.5e-3, 10e-3
y1, I1, Ez1, g1 = slit_experiment([(0, 60)], dxs, lam, 596, 4, 80)          # one slit, a = 30 mm
y2, I2, Ez2, g2 = slit_experiment([(-50, 20), (50, 20)], dxs, lam, 596, 4, 80)  # two slits, a = 10 mm, d = 50 mm
D, a1, a2, d2 = g1["D"], 60 * dxs, 20 * dxs, 100 * dxs
# The slit openings span an odd number of free nodes plus one; the pattern centre sits half a cell off zero.
y1c, y2c = y1 + 0.5 * dxs, y2 + 0.5 * dxs
print(f"grid {Ez2.shape[0]} x {Ez2.shape[1]}, {g2['n_steps']} steps per run, D = {D * 1e3:.0f} mm")

In [ ]:
def fraunhofer(y, D, lam, a, d=None):
    th = np.arctan(y / D)
    s = np.sin(th)
    env = np.sinc(a * s / lam) ** 2
    fringes = 1.0 if d is None else np.cos(np.pi * d * s / lam) ** 2
    return env * fringes * D / np.hypot(D, y)          # D / r: the 1/r fall of a cylindrical wave's intensity

fig, axes = plt.subplots(2, 2, figsize=(14, 9), gridspec_kw={"width_ratios": [1, 1.3]})
for row, (Ez, I, yc, g, a, d, name) in enumerate(((Ez1, I1, y1c, g1, a1, None, "one slit, a = 30 mm"),
                                                  (Ez2, I2, y2c, g2, a2, d2, "two slits, a = 10 mm, d = 50 mm"))):
    lim = np.abs(Ez[g["i_wall"] + 10:g["i_scr"]]).max()
    ax = axes[row, 0]
    ax.imshow(Ez.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="equal")
    ax.axvline(g["i_scr"], color="k", ls="--", lw=1)
    ax.set_title(f"{name}: Ez at the last step (dashed: screen)")
    ax.set_xticks([]); ax.set_yticks([])
    ax = axes[row, 1]
    ax.plot(yc * 1e3, I / I[np.abs(yc) < 2e-3].max(), label="FDTD, time-averaged Ez²")
    ax.plot(yc * 1e3, fraunhofer(yc, D, lam, a, d), "--", label="far-field formula")
    ax.set_xlim(-170, 170); ax.set_xlabel("y on the screen (mm)"); ax.set_ylabel("intensity / centre"); ax.legend()
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
from scipy.optimize import brentq

n1, p1 = zeros_of(I1, y1c)
n2, p2 = zeros_of(I2, y2c)

print("one slit: first zeros (mm)")
print(f"  FDTD {n1[0] * 1e3:+.1f}, {p1[0] * 1e3:+.1f}    far field D tan(asin(lambda/a)) = {D * np.tan(np.arcsin(lam / a1)) * 1e3:.1f}")

def exact_zero(m):
    return brentq(lambda yy: np.hypot(D, yy + d2 / 2) - np.hypot(D, yy - d2 / 2) - (m + 0.5) * lam, 0, 0.3)

print("\ntwo slits: zeros (mm)")
print("            FDTD           path difference    small angle (m + 1/2) lambda D / d")
for m, (zn, zp) in enumerate(zip(n2[:2], p2[:2])):
    print(f"  m = {m}:  {zn * 1e3:+7.2f} {zp * 1e3:+7.2f}      {exact_zero(m) * 1e3:7.2f}          {(m + 0.5) * lam * D / d2 * 1e3:7.2f}")
spacing = p2[0] - n2[0]
print(f"\nfringe spacing at the centre (distance between the two innermost zeros): {spacing * 1e3:.2f} mm")
print(f"lambda D / d = {lam * D / d2 * 1e3:.2f} mm   ratio {spacing / (lam * D / d2):.4f}")

The double-slit zeros sit about one percent outside the positions set by the path difference, for both the innermost pair and the second pair. The small-angle formula does worse at the second zero, where $\sin\theta = 0.3$ and the tangent exceeds the sine by five percent. The fringe spacing at the centre agrees with $\lambda D/d$ to about two percent.

The single slit's first zeros lie about four percent beyond the far-field value. Most of that comes from the finite screen distance: the slit is three wavelengths wide, so its two edges are at noticeably different distances from a point on the screen, and the far-field formula ignores that. The overlaid curves agree closely in shape. The one visible difference is that the outer double-slit peaks come out about ten percent higher than the formula gives. A slit one wavelength wide in a metal wall does not carry the uniform field the formula assumes, so each slit's own envelope is broader.

**Why twenty cells per wavelength.** With ten cells per wavelength, $\Delta x = 1$ mm, the same experiment puts the double-slit zeros about five percent too far out. At twenty cells they agree to about one percent. The error at ten cells is the grid, not the physics. The slits are then only ten cells wide, so the shape of each opening is crude, and the phase velocity depends on direction. A rule of thumb for FDTD is at least ten cells per wavelength for pictures and twenty or more for numbers. The next cell repeats the coarse run so you can see the difference.

In [ ]:
y2_coarse, I2_coarse, _, g_coarse = slit_experiment([(-25, 10), (25, 10)], 1e-3, lam, 298, 2, 40)
nc, pc = zeros_of(I2_coarse, y2_coarse + 0.5e-3)
print(f"ten cells per wavelength:    innermost zeros {nc[0] * 1e3:+.2f}, {pc[0] * 1e3:+.2f} mm   (path difference: +-{exact_zero(0) * 1e3:.2f} mm)")
print(f"twenty cells per wavelength: innermost zeros {n2[0] * 1e3:+.2f}, {p2[0] * 1e3:+.2f} mm")

### 3.4 A dielectric cylinder

A plane pulse meets a cylinder of radius 50 mm with $\epsilon_r = 4$, so $n = 2$. Three things happen at once in the snapshots. Part of the pulse reflects from the front face with a sign flip, as at the 1D dielectric step. The part inside moves at $c/2$, so it falls behind the part that passes outside. And the curved back face acts as a lens, which brings the transmitted wave to a bright spot behind the cylinder.

The delay is measured against a reference run with no cylinder. At the cylinder's centre the wave has crossed a radius of glass, so it should lag by $(n - 1)R/c$.

In [ ]:
nxc, nyc, xc, yc, Rc = 361, 301, 200, 150, 50
Xc, Yc = np.meshgrid(np.arange(nxc), np.arange(nyc), indexing="ij")
cylinder = (Xc - xc) ** 2 + (Yc - yc) ** 2 <= Rc**2
eps_cyl = np.where(cylinder, 4.0, 1.0)
wc = 60 * dt
pulse_c = lambda t: fd.gaussian(t, 4 * wc, wc)
sheet = (np.full(nyc - 2, 40), np.arange(1, nyc - 1))
centre_probe = [(xc, yc)]

with_cyl = fd.fdtd_2d_tm(nxc, nyc, dx, 1000, eps_r=eps_cyl, source=pulse_c, src_index=sheet,
                         probes=centre_probe, snapshot_every=200)
reference = fd.fdtd_2d_tm(nxc, nyc, dx, 1000, source=pulse_c, src_index=sheet, probes=centre_probe)
lag = with_cyl["t"][np.argmax(with_cyl["probe_E"][:, 0])] - reference["t"][np.argmax(reference["probe_E"][:, 0])]
print(f"lag at the centre: {lag / dt:.0f} steps   (n - 1) R / c = {Rc * dx / fd.C0 / dt:.0f} steps")

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
theta = np.linspace(0, 2 * np.pi, 200)
for ax, k in zip(axes, range(1, 5)):
    snap = with_cyl["snapshots"][k]
    lim = np.abs(snap[30:-30, 30:-30]).max()
    ax.imshow(snap.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.plot(xc + Rc * np.cos(theta), yc + Rc * np.sin(theta), "k", lw=0.8)
    ax.set_title(f"step {200 * (k + 1)}"); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("a plane pulse meets a dielectric cylinder (eps_r = 4)")
plt.tight_layout(); plt.show()

At the centre the lag matches $(n - 1)R/c$ to within a step. Closer to the back face a single probe stops being a clean test: waves refracted at different points of the curved front converge there and arrive earlier than the straight ray through the axis, which is the lens at work. The faint patches near the top and bottom edges of the snapshots come from the ends of the source column, where it runs into the absorbing frame; they stay well away from the axis where the delay is measured.

## 4. Verify

Four checks from this notebook, each with a tolerance and its reason.

1. **Speed.** The delay between radii of 40 and 80 cells gives $c$ within 2 percent. A diagonal probe at nearly the same radius arrives within 3 steps of the time its radius predicts.
2. **Cylindrical spreading.** The peak amplitudes at 40 and 80 cells differ by $\sqrt2$ within 3 percent.
3. **The frame.** After 1400 steps the pulse has left. What remains on the grid is less than 0.5 percent of the peak that passed $r = 40$ cells, which bounds what the frame reflects.
4. **Double-slit fringes.** The spacing of the innermost zeros is within 10 percent of $\lambda D/d$. The tolerance allows for the finite screen distance and the angles, and the measured agreement is far closer.

The plot shows what the frame leaves behind, on a logarithmic scale.

In [ ]:
late = fd.fdtd_2d_tm(n, n, dx, 1400, source=pulse, src_index=c0)
residual = np.abs(late["Ez"]).max() / amp_axis[2]

checks = {
    "speed from two radii / c": v / fd.C0,
    "diagonal arrival minus prediction (steps)": (t_diag[4] - t_axis[4]) / dt - (r_diag[4] - radii[4]) * dx / fd.C0 / dt,
    "amplitude ratio r = 40 / r = 80, over sqrt 2": amp_axis[2] / amp_axis[4] / np.sqrt(2),
    "field left after 1400 steps / peak at r = 40": residual,
    "fringe spacing / (lambda D / d)": spacing / (lam * D / d2),
}
for k_, v_ in checks.items():
    print(f"{k_:48s} {v_: .4g}")

assert abs(v / fd.C0 - 1) < 0.02
assert abs(checks["diagonal arrival minus prediction (steps)"]) < 3
assert abs(amp_axis[2] / amp_axis[4] / np.sqrt(2) - 1) < 0.03
assert residual < 5e-3
assert abs(spacing / (lam * D / d2) - 1) < 0.10

plt.imshow(np.log10(np.abs(late["Ez"]) / amp_axis[2] + 1e-12).T, origin="lower", cmap="viridis", vmin=-8, vmax=-2)
plt.colorbar(label="log10 |Ez| / peak at r = 40")
plt.title("what is left after 1400 steps"); plt.xlabel("x (cells)"); plt.ylabel("y (cells)")
plt.show()

## 5. Explore and rebuild

**Explore.** The frame is a compromise between thickness and strength. Run the point source with `n_layer` of 5, 10, 20 and 40 cells and `layer_strength` from 0.25 to 4. For each pair, record the largest field that returns to the source after the pulse has left it, relative to the outgoing peak. A layer too weak lets the wave reach the PEC wall and come back. A layer too strong reflects at its own front edge, because the conductivity jumps too fast from one cell to the next. Find the best strength for each thickness and explain why thicker layers are more forgiving.

A second option: use a column of soft sources whose phase advances along $y$ to launch a plane wave at an angle, and let it fall on a dielectric half-space with $\epsilon_r = 4$. Measure the direction of the transmitted wavefronts and test Snell's law, $\sin\theta_1 = 2\sin\theta_2$.

**Rebuild.** Close this notebook. From memory, write the 2D TM update for $H_x$, $H_y$ and $E_z$ on the Yee grid with the correct signs and staggering. Run it in vacuum with PEC walls and a point source, and check it against `emlib.fdtd.fdtd_2d_tm` with `n_layer=0`. Then add the Poynting vector and repeat the energy check of section 3.2.